In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

import nibabel as nib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    roc_auc_score
)

import pandas as pd
import numpy as np
from pathlib import Path
import re

In [ ]:
def create_oasis_dataframe(nii_root, csv_path="oasis_cross-sectional.csv"):

    nii_root = Path(nii_root)

    df = pd.read_csv(csv_path)

    df["subject_id"] = df["ID"].str.extract(
        r"^(OAS1_\d+)_MR\d+$"
    )

    df["session"] = df["ID"].str.extract(
        r"_(MR\d+)$"
    )

    df["label"] = (df["CDR"] > 0).astype(int)

    df["class_name"] = df["label"].map({
        0: "Non-demented",
        1: "Demented"
    })

    nii_files = list(nii_root.glob("*.nii")) + list(
        nii_root.glob("*.nii.gz")
    )

    print(f"Found {len(nii_files)} NIfTI files")

    nii_map = {}

    for path in nii_files:

        match = re.search(
            r"(OAS1_\d+_MR\d+)",
            path.name
        )

        if match:
            scan_id = match.group(1)
            nii_map[scan_id] = str(path)

    df["nii_path"] = df["ID"].map(nii_map)

    matched = df["nii_path"].notna().sum()
    missing = df["nii_path"].isna().sum()

    print(f"Matched MRI files: {matched}")
    print(f"Missing MRI files: {missing}")

    if missing > 0:
        print("\nMissing MRI files:")
        print(df.loc[
            df["nii_path"].isna(),
            ["ID", "subject_id", "session"]
        ])

    return df

In [3]:
df = create_oasis_dataframe(
    nii_root="raw/oasis/OASIS",
    csv_path="raw/oasis_cross-sectional.csv"
)

cnn_df = df[df["session"] == "MR1"].copy()

cnn_df = cnn_df[
    cnn_df["nii_path"].notna()
].reset_index(drop=True)

print(cnn_df.shape)
print(cnn_df["label"].value_counts())

Found 436 NIfTI files
Matched MRI files: 436
Missing MRI files: 0
(416, 17)
label
0    316
1    100
Name: count, dtype: int64


In [4]:
train_df, test_df = train_test_split(
    cnn_df,
    test_size=0.20,
    stratify=cnn_df["label"],
    random_state=42
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.20,
    stratify=train_df["label"],
    random_state=42
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

Train: (265, 17)
Validation: (67, 17)
Test: (84, 17)

Train labels:
label
0    201
1     64
Name: count, dtype: int64

Validation labels:
label
0    51
1    16
Name: count, dtype: int64

Test labels:
label
0    64
1    20
Name: count, dtype: int64


In [ ]:
class OASISDataset(Dataset):

    def __init__(self, dataframe, target_size=(128, 128, 128)):
        self.df = dataframe.reset_index(drop=True)
        self.target_size = target_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img = nib.load(row["nii_path"])

        volume = img.get_fdata().astype(np.float32)

        volume = np.nan_to_num(volume)

        nonzero = volume[volume != 0]

        mean = nonzero.mean()
        std = nonzero.std()

        volume = (volume - mean) / (std + 1e-8)

        volume = torch.tensor(volume, dtype=torch.float32)
        volume = volume.unsqueeze(0)

        volume = volume.unsqueeze(0)

        # Resize
        volume = F.interpolate(
            volume,
            size=self.target_size,
            mode="trilinear",
            align_corners=False
        )

        volume = volume.squeeze(0)

        label = torch.tensor(
            row["label"],
            dtype=torch.long
        )

        return volume, label

In [6]:
train_dataset = OASISDataset(train_df)
val_dataset = OASISDataset(val_df)
test_dataset = OASISDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [ ]:
class MRI3DCNN(nn.Module):

    def __init__(self, num_classes=2):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv3d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm3d(16),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm3d(128),
            nn.ReLU()
        )

        self.pool = nn.AdaptiveAvgPool3d(1)

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = self.classifier(x)

        return x

In [8]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using:", device)

Using: mps


In [9]:
model = MRI3DCNN(num_classes=2)
model = model.to(device)

print(model)

MRI3DCNN(
  (features): Sequential(
    (0): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (5): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (9): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv3d(64, 128, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (13): BatchNorm3d(128

In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [11]:
class_counts = train_df["label"].value_counts().sort_index()

class_weights = len(train_df) / (
    2 * class_counts
)

class_weights = torch.tensor(
    class_weights.values,
    dtype=torch.float32
).to(device)

print(class_weights)

tensor([0.6592, 2.0703], device='mps:0')


In [12]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

In [13]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for X, y in loader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(X)

        loss = criterion(outputs, y)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * X.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == y).sum().item()
        total += y.size(0)

    loss = running_loss / total
    accuracy = correct / total

    return loss, accuracy

In [14]:
def evaluate(model, loader, criterion, device):

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():

        for X, y in loader:

            X = X.to(device)
            y = y.to(device)

            outputs = model(X)

            loss = criterion(outputs, y)

            probabilities = torch.softmax(
                outputs,
                dim=1
            )

            predictions = outputs.argmax(dim=1)

            running_loss += loss.item() * X.size(0)

            correct += (
                predictions == y
            ).sum().item()

            total += y.size(0)

            all_labels.extend(
                y.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_probabilities.extend(
                probabilities[:, 1]
                .cpu()
                .numpy()
            )

    loss = running_loss / total
    accuracy = correct / total

    return (
        loss,
        accuracy,
        np.array(all_labels),
        np.array(all_predictions),
        np.array(all_probabilities)
    )

In [15]:
num_epochs = 40

best_val_loss = float("inf")

history = []

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    (
        val_loss,
        val_acc,
        y_val,
        pred_val,
        prob_val
    ) = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    # Early stopping
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "best_oasis_cnn.pt"
        )

Epoch 01/40 | Train Loss: 0.6060 | Train Acc: 0.7057 | Val Loss: 0.6023 | Val Acc: 0.7313
Epoch 02/40 | Train Loss: 0.5624 | Train Acc: 0.7283 | Val Loss: 0.6092 | Val Acc: 0.6269
Epoch 03/40 | Train Loss: 0.5788 | Train Acc: 0.7321 | Val Loss: 0.5525 | Val Acc: 0.7463
Epoch 04/40 | Train Loss: 0.5801 | Train Acc: 0.7396 | Val Loss: 0.5440 | Val Acc: 0.6716
Epoch 05/40 | Train Loss: 0.5689 | Train Acc: 0.7434 | Val Loss: 0.5424 | Val Acc: 0.7463
Epoch 06/40 | Train Loss: 0.5431 | Train Acc: 0.7962 | Val Loss: 0.5424 | Val Acc: 0.7015
Epoch 07/40 | Train Loss: 0.5832 | Train Acc: 0.7547 | Val Loss: 0.5542 | Val Acc: 0.6716
Epoch 08/40 | Train Loss: 0.6144 | Train Acc: 0.7736 | Val Loss: 0.5510 | Val Acc: 0.7612
Epoch 09/40 | Train Loss: 0.5735 | Train Acc: 0.7774 | Val Loss: 0.6767 | Val Acc: 0.6418
Epoch 10/40 | Train Loss: 0.5049 | Train Acc: 0.7774 | Val Loss: 0.5740 | Val Acc: 0.6418
Epoch 11/40 | Train Loss: 0.5857 | Train Acc: 0.7698 | Val Loss: 0.6387 | Val Acc: 0.5970
Epoch 12/4

KeyboardInterrupt: 

In [16]:
model.load_state_dict(
    torch.load(
        "best_oasis_cnn.pt",
        map_location=device
    )
)

<All keys matched successfully>

In [17]:
(
    test_loss,
    test_acc,
    y_test,
    pred_test,
    prob_test
) = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

Test Loss: 0.4711
Test Accuracy: 0.8571


In [18]:
print(
    classification_report(
        y_test,
        pred_test,
        target_names=[
            "Non-demented",
            "Demented"
        ]
    )
)

              precision    recall  f1-score   support

Non-demented       0.87      0.95      0.91        64
    Demented       0.79      0.55      0.65        20

    accuracy                           0.86        84
   macro avg       0.83      0.75      0.78        84
weighted avg       0.85      0.86      0.85        84



In [19]:
print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_test,
        pred_test
    )
)

Balanced Accuracy: 0.7515625


In [20]:
auc = roc_auc_score(
    y_test,
    prob_test
)

print("ROC-AUC:", auc)

ROC-AUC: 0.86328125


In [21]:
cm = confusion_matrix(
    y_test,
    pred_test
)

print(cm)

[[61  3]
 [ 9 11]]


In [22]:
torch.save(
    model.state_dict(),
    "oasis_3d_cnn_new.pth"
)

print("Model saved successfully!")

Model saved successfully!
